In [ ]:
import os
import re
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm

# ----------------
# Paths 
# ----------------
DATASET_ROOT = "./WebNLG_CO"  # folder containing xml files (recursively)

ENTITY_CSVS = {
    "ca": "./Registry_triples/entity_translations_ca_output_token.csv"
}

RELATION_CSVS = {
    "ca": "./Registry_triples/relation_translations_ca_with_output_token.csv"
}

OVERWRITE_EXISTING = True  # True to regenerate even if target triplesets already have triples

TRIPLE_SPLIT = " | "  # WebNLG format

In [17]:
# ----------------
# Helpers
# ----------------
def iter_xml_files(root_dir: str):
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith(".xml"):
                yield os.path.join(dirpath, fn)

def indent_xml(elem: ET.Element, level: int = 0):
    i = "\n" + level * "  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for child in elem:
            indent_xml(child, level + 1)
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
    else:
        if level and (not elem.tail or not elem.tail.strip()):
            elem.tail = i

def write_xml_atomic(tree: ET.ElementTree, path: str):
    tmp_path = path + ".tmp"
    root = tree.getroot()
    indent_xml(root, 0)
    tree.write(tmp_path, encoding="utf-8", xml_declaration=True)
    os.replace(tmp_path, path)

def ensure_tripleset_node(entry: ET.Element, tag: str, insert_after: str | None = None) -> ET.Element:
    existing = entry.find(tag)
    if existing is not None:
        return existing

    new_node = ET.Element(tag)

    if insert_after:
        after_node = entry.find(insert_after)
        if after_node is not None:
            children = list(entry)
            idx = children.index(after_node)
            entry.insert(idx + 1, new_node)
            return new_node

    sp = entry.find("spanishtripleset")
    if sp is not None:
        children = list(entry)
        idx = children.index(sp)
        entry.insert(idx + 1, new_node)
        return new_node

    entry.append(new_node)
    return new_node

def tripleset_has_triples(node: ET.Element) -> bool:
    return any((c.text or "").strip() for c in list(node))

def clear_children(node: ET.Element):
    for c in list(node):
        node.remove(c)

def parse_triple_line(line: str):
    parts = [p.strip() for p in line.split(TRIPLE_SPLIT, 2)]
    if len(parts) != 3:
        parts = [p.strip() for p in line.split("|", 2)]
    if len(parts) != 3:
        raise ValueError(f"Bad triple: {line!r}")
    return parts[0], parts[1], parts[2]


In [ ]:

def build_map_from_csv(path: str):
    df = pd.read_csv(path)

    req = {"source_token", "output_token"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {sorted(missing)}")

    # keep only non-empty mappings
    df["source_token"] = df["source_token"].astype(str).str.strip()
    df["output_token"] = df["output_token"].astype(str).str.strip()
    df = df[(df["source_token"] != "") & (df["output_token"] != "")].copy()

    return dict(zip(df["source_token"], df["output_token"]))
    

entity_map = {lang: build_map_from_csv(path) for lang, path in ENTITY_CSVS.items()}
rel_map    = {lang: build_map_from_csv(path) for lang, path in RELATION_CSVS.items()}

print("Loaded maps:")
for lg in ("ca"):
    print(lg, "entities:", len(entity_map[lg]), "| relations:", len(rel_map[lg]))


# ----------------
# Rebuild XML triplesets
# ----------------
# ----------------
# Rebuild XML triplesets
# - Source: English modifiedtripleset / mtriple
# - Placement: after spanishtripleset
# - Maps: source_token -> output_token
# ----------------
LANG_SPECS = {
    "ca": ("catalantripleset", "ctriple")
}

LANG_ORDER = ["ca"]

def insert_after(parent: ET.Element, ref_child: ET.Element, new_child: ET.Element):
    children = list(parent)
    idx = children.index(ref_child)
    parent.insert(idx + 1, new_child)

def ensure_positioned_tripleset(entry: ET.Element, lang: str) -> ET.Element:
    tripleset_tag, _ = LANG_SPECS[lang]

    node = entry.find(tripleset_tag)
    if node is None:
        node = ET.Element(tripleset_tag)

    # remove if already present so we can reinsert in the right position
    for child in list(entry):
        if child is node:
            entry.remove(child)
            break

    spanish_node = entry.find("spanishtripleset")
    if spanish_node is None:
        entry.append(node)
        return node

    prev = spanish_node
    for lg in LANG_ORDER:
        existing = entry.find(LANG_SPECS[lg][0])
        if lg == lang:
            insert_after(entry, prev, node)
            return node
        elif existing is not None:
            prev = existing

    insert_after(entry, spanish_node, node)
    return node

def rebuild_triplesets_all_langs(
    dataset_root: str,
    entity_map,
    rel_map,
    overwrite_existing: bool = False
):
    xml_files = sorted(iter_xml_files(dataset_root))
    changed_files = 0
    total_entries = 0
    total_triples_written = {lang: 0 for lang in LANG_SPECS}

    for xp in tqdm(xml_files, desc="Writing catalan triplesets", unit="file"):
        try:
            tree = ET.parse(xp)
        except Exception:
            continue

        root = tree.getroot()
        entries_parent = root.find("entries")
        if entries_parent is None:
            continue

        file_changed = False

        for entry in entries_parent.findall("entry"):
            total_entries += 1

            # SOURCE = English modified triples
            en_node = entry.find("modifiedtripleset")
            if en_node is None:
                continue

            source_triples = en_node.findall("mtriple")
            if not source_triples:
                continue

            for lang in LANG_ORDER:
                tripleset_tag, triple_tag = LANG_SPECS[lang]
                tgt_node = ensure_positioned_tripleset(entry, lang)

                if not overwrite_existing and tripleset_has_triples(tgt_node):
                    continue

                clear_children(tgt_node)
                wrote_any = False

                for st in source_triples:
                    line = (st.text or "").strip()
                    if not line:
                        continue

                    try:
                        s_en, p_en, o_en = parse_triple_line(line)
                    except Exception:
                        continue

                    # translate with source_token -> output_token
                    s_tgt = entity_map[lang].get(s_en, s_en)
                    p_tgt = rel_map[lang].get(p_en, p_en)
                    o_tgt = entity_map[lang].get(o_en, o_en)

                    new_line = f"{s_tgt}{TRIPLE_SPLIT}{p_tgt}{TRIPLE_SPLIT}{o_tgt}"

                    el = ET.Element(triple_tag)
                    el.text = new_line
                    tgt_node.append(el)

                    total_triples_written[lang] += 1
                    wrote_any = True

                if wrote_any:
                    file_changed = True

        if file_changed:
            write_xml_atomic(tree, xp)
            changed_files += 1

    print("Done.")
    print("Changed files:", changed_files, "/", len(xml_files))
    print("Entries scanned:", total_entries)
    for lang in LANG_ORDER:
        print(f"{lang} triples written:", total_triples_written[lang])

Loaded maps:
ca entities: 3624 | relations: 411
gl entities: 3624 | relations: 411
eu entities: 3624 | relations: 411


In [19]:
rebuild_triplesets_all_langs(
    DATASET_ROOT,
    entity_map=entity_map,
    rel_map=rel_map,
    overwrite_existing=OVERWRITE_EXISTING,
)

Writing co-official triplesets:   0%|          | 0/178 [00:00<?, ?file/s]

Done.
Changed files: 178 / 178
Entries scanned: 16687
ca triples written: 48879
gl triples written: 48879
eu triples written: 48879


In [ ]:
import os
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm
from collections import defaultdict

DATASET_ROOT = "./WebNLG_ES"  # change if needed

def iter_xml_files(root_dir: str):
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith(".xml"):
                yield os.path.join(dirpath, fn)

def count_triples(node):
    if node is None:
        return 0
    return sum(1 for c in list(node) if (c.text or "").strip())

stats = {
    "total_entries": 0,
    "missing_spanish": 0,
    "missing_ca": 0,
    "missing_gl": 0,
    "missing_eu": 0,
    "empty_ca": 0,
    "empty_gl": 0,
    "empty_eu": 0,
    "count_mismatch": 0,
}

problems = []

xml_files = sorted(iter_xml_files(DATASET_ROOT))

for xp in tqdm(xml_files, desc="Checking XML", unit="file"):
    try:
        tree = ET.parse(xp)
    except Exception:
        continue

    root = tree.getroot()
    entries_parent = root.find("entries")
    if entries_parent is None:
        continue

    for entry in entries_parent.findall("entry"):
        stats["total_entries"] += 1

        sp = entry.find("spanishtripleset")
        ca = entry.find("catalantripleset")

        # Missing nodes
        if sp is None:
            stats["missing_spanish"] += 1
        if ca is None:
            stats["missing_ca"] += 1

        # Skip detailed checks if spanish missing
        if sp is None:
            problems.append((xp, "missing_spanish"))
            continue

        n_sp = count_triples(sp)
        n_ca = count_triples(ca)
        n_gl = count_triples(gl)
        n_eu = count_triples(eu)

        # Empty triplesets
        if ca is not None and n_ca == 0:
            stats["empty_ca"] += 1

        # Count mismatch
        if not (n_sp == n_ca == n_gl == n_eu):
            stats["count_mismatch"] += 1
            problems.append((xp, f"mismatch sp={n_sp} ca={n_ca} gl={n_gl} eu={n_eu}"))

# -------------------------
# Report
# -------------------------
print("\n===== SUMMARY =====")
for k, v in stats.items():
    print(f"{k}: {v}")

print("\nEntries with problems:", len(problems))

# Show first 10 problematic entries
print("\nSample problems (max 10):")
for p in problems[:10]:
    print(p)

Checking XML:   0%|          | 0/191 [00:00<?, ?file/s]


===== SUMMARY =====
total_entries: 18671
missing_spanish: 0
missing_ca: 0
missing_gl: 0
missing_eu: 0
empty_ca: 0
empty_gl: 0
empty_eu: 0
count_mismatch: 0

Entries with problems: 0

Sample problems (max 10):
